# Exploración de datos. Ejercicios

Francisco Jurado Monroy <francisco.jurado@uam.es>

Universidad Autónoma de Madrid

<hr/>

## Análisis de datos de Fórmula 1

En primer lugar, cargaremos los datos publicados en una página web con información sobre la clasificación en las diferentes carreras disputadas.

In [ ]:
import pandas as pd
df = pd.read_html('https://www.statsf1.com/es/2025.aspx',
                  storage_options={"User-Agent": "Mozilla/5.0"})[0]

In [ ]:
df.head()

Eliminaremos las filas y columnas que no aportan nada.

In [ ]:
# eliminamos la fina 4 que no contiene datos
df = df.drop(index=4)

# También eliminaremos la primera columna 'Pilotos' que se encuentra repetida y solo muestra la posición de los pilotos
df = df.drop(columns=0)

Usaremos la primera fila como nombres de columnas

In [ ]:
new_header = df.iloc[0]
df = df[1:]
df.columns = new_header

Eliminaremos la columna de puntos totales 'Pts'. No nos fiamos... ya lo calcularemos nosotros.

In [ ]:
df = df.drop(columns=['Pts'])

Tratemos las celdas que no tienen puntos y los valores nulos

In [ ]:
# reemplazamos los valores con "-" por 0
df = df.transform(lambda x: x.replace('-', 0))

# eliminamos las columnas que solo tienen NaN
df = df.dropna(axis=1, how='all') # axis=1 indica que se eliminan columnas, y how='all' indica que se eliminan solo si todos los valores son NaN

Veamos el tipo de datos de cada columna

In [ ]:
df.dtypes

Excepto la columna 'Pilotos', todas las demás columnas deben ser de tipo numérico.

Pero ojo, no puede convertirse a Int64 porque hay valores NaN, pero sí a float.

Esto es porque los valores NaN se representan como float en pandas

In [ ]:
for col in df.columns[1:]:
    df[col] = pd.to_numeric(df[col]) # equivale a df[col] = df[col].astype(float)

In [ ]:
df.dtypes

Ya podemos guardar el dataframe limpio en un archivo CSV para su uso posterior

In [ ]:
df.to_csv('f1_2025_cleaned.csv', index=False)

Dibujaremos un gráfico de líneas con la evolución de los puntos de los 3 primeros pilotos del campeonato a lo largo de las carreras disputadas hasta el momento.

In [ ]:
df_top3 = df.head(3)
df_top3 = df_top3.set_index('Pilotos')
df_top3.T.plot.line(figsize=(10,6), marker='o', title='Evolución de puntos de los 3 primeros pilotos en el campeonato de F1 2025', xlabel='Carreras', ylabel='Puntos')

Calculemos el número total de puntos obtenidos por cada piloto en las carreras disputadas hasta el momento

In [ ]:
df['Total_Points'] = df.iloc[:, 1:].sum(axis=1)
df[['Pilotos', 'Total_Points']].sort_values(by='Total_Points', ascending=False)

¿Cuál es la carrera en la que más pilotos han puntuado?

In [ ]:
points_per_race = (df.iloc[:, 1:-2] > 0).sum()
points_per_race.idxmax()    

¿Cuál es el piloto que menos avandonos ha tenido?

In [ ]:
df['Retirements'] = (df.iloc[:, 1:-3] == 0).sum(axis=1)
df[['Pilotos', 'Retirements']].sort_values(by='Retirements')

¿Cuántos han puntuado en todas las carreras disputadas hasta el momento? O lo que es lo mismo, ¿cuántos han tenido cero abandonos?

In [ ]:
df.loc[df['Retirements'] == 0, 'Pilotos']

¿Y cuáles son esos pilotos que menos retiros han tenido?

In [ ]:
df.loc[df['Retirements'] == df['Retirements'].min(), 'Pilotos']

¿Cuántos de los pilotos que han tenido menos retiros están entre los 5 primeros del campeonato?

In [ ]:
top10_pilots = df.head(5)['Pilotos']
min_retirement_pilots = df.loc[df['Retirements'] == df['Retirements'].min(), 'Pilotos']
len(set(top10_pilots) & set(min_retirement_pilots))

que precisamente son...

In [ ]:
set(top10_pilots) & set(min_retirement_pilots)